In [65]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

# Load CSV
df = pd.read_csv("../data/telco_churn.csv")

df.head()


FileNotFoundError: [Errno 2] No such file or directory: '../data/telco_churn.csv'

In [ ]:
df.shape, df.dtypes


In [ ]:
df["Churn"].value_counts(normalize=True)


In [ ]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# How many became NaN?
df["TotalCharges"].isna().sum()


In [ ]:
df = df.dropna(subset=["TotalCharges"])
df.shape


In [ ]:
df.columns = df.columns.str.strip().str.replace(" ", "_")
df.head()


In [ ]:
df.info()


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Churn", palette="Set2")
plt.title("Churn Distribution")
plt.show()

df["Churn"].value_counts(normalize=True)


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Churn by Contract Type")
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
sns.kdeplot(data=df, x="tenure", hue="Churn", common_norm=False)
plt.title("Churn Probability by Tenure")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges vs Churn")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x="InternetService", hue="Churn")
plt.title("Churn by Internet Service Type")
plt.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.select_dtypes(include=['int64','float64']).corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
service_cols = [
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies"
]

df["ServiceCount"] = df[service_cols].apply(lambda row: 
                    sum(row == "Yes") if "Yes" in row.values else 0, axis=1)

df["ServiceCount"].head()


In [ ]:
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=["0–1 year", "1–2 years", "2–4 years", "4–6 years"]
)


In [ ]:
df["ChargesTier"] = pd.qcut(df["MonthlyCharges"], q=4, labels=["Low", "Medium", "High", "Very High"])


In [ ]:
df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)


In [ ]:
df["IsAutoPay"] = df["PaymentMethod"].str.contains("automatic", case=False).astype(int)


In [ ]:
df["SeniorAlone"] = (
    (df["SeniorCitizen"] == 1) & 
    (df["Partner"] == "No") & 
    (df["Dependents"] == "No")
).astype(int)


In [ ]:
df["CLTV"] = df["MonthlyCharges"] * df["tenure"]


In [ ]:
df_model = df.drop(columns=["customerID"])


In [ ]:
from sklearn.model_selection import train_test_split

# Work on the engineered dataframe
data = df_model.copy()

# Encode target: Yes->1, No->0
data["Churn"] = (data["Churn"] == "Yes").astype(int)

# Separate features and target
X = data.drop(columns=["Churn"])
y = data["Churn"]

X.shape, y.shape, y.mean()


In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features, categorical_features


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape, y_train.mean(), y_test.mean()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Preprocess: scale numeric (optional later), one-hot encode categoricals
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",   # handles imbalance a bit
    solver="liblinear"
)

clf_logreg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", log_reg)
])

clf_logreg

In [ ]:
clf_logreg.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred = clf_logreg.predict(X_test)
y_proba = clf_logreg.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print()
print(classification_report(y_test, y_pred, digits=3))

confusion_matrix(y_test, y_pred)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

clf_rf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf)
])

clf_rf.fit(X_train, y_train)

rf_proba = clf_rf.predict_proba(X_test)[:, 1]
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_proba))


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    random_state=42,
    max_depth=6,
    learning_rate=0.05,
    max_iter=300
)

# same preprocess (one-hot + numeric passthrough)
clf_hgb = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", hgb)
])

clf_hgb.fit(X_train, y_train)

hgb_proba = clf_hgb.predict_proba(X_test)[:, 1]
print("HistGradientBoosting ROC-AUC:", roc_auc_score(y_test, hgb_proba))


In [ ]:
results = {
    "LogReg": roc_auc_score(y_test, y_proba),
    "RandomForest": roc_auc_score(y_test, rf_proba),
    "HistGB": roc_auc_score(y_test, hgb_proba),
}

pd.DataFrame(results, index=["ROC_AUC"]).T.sort_values("ROC_AUC", ascending=False)


In [ ]:
import shap

# Get preprocessed feature matrix
X_train_transformed = preprocess.fit_transform(X_train)

# Get feature names after one-hot encoding
ohe = preprocess.named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_features)
all_feature_names = np.concatenate([cat_feature_names, numeric_features])

# Fit standalone logistic model on transformed data
logreg_shap = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear"
)
logreg_shap.fit(X_train_transformed, y_train)

# SHAP explainer
explainer = shap.LinearExplainer(logreg_shap, X_train_transformed)
shap_values = explainer.shap_values(X_train_transformed)

# SHAP summary plot
shap.summary_plot(shap_values, X_train_transformed, feature_names=all_feature_names)


## Business Recommendations

Based on the churn model and SHAP analysis:

1. Prioritise retention campaigns for **new customers (< 12 months tenure)**.
2. Incentivise **month-to-month users** to switch to annual or two-year contracts.
3. Offer **discounts or bundles** to high MonthlyCharges + Fiber optic customers.
4. Promote **AutoPay and multi-service bundles** to reduce churn risk.
5. Improve adoption of **OnlineSecurity and TechSupport**, especially for fiber users.